# Parallel & Concurrent Programming

## Measure before optimizing

The `duplicates` function below returns the items that appear several times in a list.

1. With [`timeit.timeit`](https://docs.python.org/3/library/timeit.html#timeit.timeit), measure the execution time of `duplicates(numbers)`.
2. With [`cProfile.run`](https://docs.python.org/3/library/profile.html#profile.run), find the call in which the function spends most of its time.
3. Write a faster version, check that it returns the same items and measure the gain with `timeit`.

In [ ]:
import random

numbers = [random.randrange(5_000) for _ in range(5_000)]


def duplicates(values: list[int]) -> list[int]:
  result = []
  for value in values:
    if values.count(value) > 1 and value not in result:
      result.append(value)
  return result


# Your code here

### Solution

In [ ]:
import collections
import cProfile
import timeit

duration = timeit.timeit("duplicates(numbers)", globals=globals(), number=3) / 3
print(f"duplicates: {duration:.3f}s")

cProfile.run("duplicates(numbers)", sort="tottime")


def fast_duplicates(values: list[int]) -> list[int]:
  return [value for value, n in collections.Counter(values).items() if n > 1]


assert sorted(fast_duplicates(numbers)) == sorted(duplicates(numbers))
duration = timeit.timeit("fast_duplicates(numbers)", globals=globals(), number=3) / 3
print(f"fast_duplicates: {duration:.5f}s")

The profile shows that almost all the time is spent in `list.count`, called for every item: each call scans the whole list again, so the algorithm is quadratic. [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) counts every occurrence in a single pass.

## Calling a Function with an Argument in Parallel

Call the function `f` that gives information about the process executing it in parallel using a `multiprocessing.Pool`.

In [ ]:
import multiprocessing
import os


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"Parent process ID: {os.getppid()}")
  print(f"Process ID: {os.getpid()}")


# Your code here

### Solution

In [ ]:
import multiprocessing
import os


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"Parent process ID: {os.getppid()}")
  print(f"Process ID: {os.getpid()}")


with multiprocessing.Pool() as pool:
  pool.map(f, range(10))

## Calling a Function in Parallel with Constant Arguments

Use [`functools.partial`](https://docs.python.org/3/library/functools.html#functools.partial) and adapt the previous code to call `f` with `n` ranging from 0 to 9 and `verbose` always set to `False`.

In [ ]:
import functools
import multiprocessing
import os


def f(n: int, verbose: bool) -> int:
  if verbose:
    print(f"Process {n}")
    print(f"Parent process ID: {os.getppid()}")
    print(f"Process ID: {os.getpid()}")
  return n * 2


# Your code here

### Solution

In [ ]:
import functools
import multiprocessing
import os


def f(n: int, verbose: bool) -> int:
  if verbose:
    print(f"Process {n}")
    print(f"Parent process ID: {os.getppid()}")
    print(f"Process ID: {os.getpid()}")
  return n * 2


with multiprocessing.Pool() as pool:
  results = pool.map(functools.partial(f, verbose=False), range(10))

print(results)

## Downloading Multiple Files Simultaneously, with a Single Argument

Use a `multiprocessing.pool.ThreadPool` to download multiple files simultaneously. Initially, we will download the Wikipedia random page 10 times: `https://en.wikipedia.org/wiki/Special:Random`. The parallelized function will simply retrieve an integer and store the download result in an `articles` folder under `{n}.html` if `n` is the integer.

For file downloading, you can use the following code:

```python
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)
```

The `User-Agent` header is required: Wikipedia, like other websites, rejects the default `urllib` agent (HTTP error 403).

where `url` is the URL to download and `path` is the path to write the file.

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request

articles = pathlib.Path("articles")
articles.mkdir(exist_ok=True)
url = "https://en.wikipedia.org/wiki/Special:Random"


# Your code here

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request

articles = pathlib.Path("articles")
articles.mkdir(exist_ok=True)
url = "https://en.wikipedia.org/wiki/Special:Random"


def download_article(n: int) -> None:
  # Wikipedia and Python Tutor reject the default urllib agent (error 403)
  request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
  with (
    urllib.request.urlopen(request) as response,
    (articles / f"{n}.html").open("wb") as fh,
  ):
    shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  results = pool.map(download_article, range(10))

## Downloading Multiple Files Simultaneously, with Two Arguments

Similar to the previous exercise, we want to download multiple pages at the same time. This time we want to give our worker a URL and a path to write to, rather than just an integer.

Adapt the previous code to download `to_download`.

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request

downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
  ("http://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
)


# Your code here

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request

downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
  ("http://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
)


def download(url: str, path: pathlib.Path) -> None:
  # Wikipedia and Python Tutor reject the default urllib agent (error 403)
  request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
  with urllib.request.urlopen(request) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  pool.starmap(download, to_download)

## Concurrent downloads with `asyncio`

Reuse `to_download` and the `download` function from the previous exercise, and download the pages concurrently with `asyncio`: [`asyncio.to_thread`](https://docs.python.org/3/library/asyncio-task.html#asyncio.to_thread) runs a blocking function (such as `download`) without blocking the event loop, and [`asyncio.gather`](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather) waits for several coroutines at once. Display the total download time.

Beware: in Colab (as in Jupyter), an event loop is already running, so `asyncio.run(main())` raises a `RuntimeError`. Write `await main()` directly in the cell; in a script, you would use `asyncio.run(main())`.

In [ ]:
# Your code here

### Solution

In [ ]:
import asyncio
import time


async def main() -> None:
  start = time.perf_counter()
  await asyncio.gather(
    *(asyncio.to_thread(download, url, path) for url, path in to_download)
  )
  print(f"{len(to_download)} pages downloaded in {time.perf_counter() - start:.2f}s")


await main()

## Creating a Producer/Consumer Architecture

The following code makes two processes communicate through a [`multiprocessing.Queue`](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.Queue): the producer puts items in it, the consumer takes them out and prints them.

Run it, then answer: why does the producer put `None` at the end? Then modify the code to start two consumers instead of one.

In [ ]:
import multiprocessing


def consumer(queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(item)


def producer(queue: multiprocessing.Queue) -> None:
  for i in range(10):
    queue.put(i)
  queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumer = multiprocessing.Process(target=consumer, args=(queue,))
  producer = multiprocessing.Process(target=producer, args=(queue,))
  consumer.start()
  producer.start()
  consumer.join()
  producer.join()

In [ ]:
# Your code here

### Solution

`None` is a sentinel value: it tells the consumer that no more items will come. Without it, the consumer would block forever on `queue.get()`. With several consumers, each one must receive its own sentinel:

In [ ]:
import multiprocessing


def consumer(name: str, queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(f"{name}: {item}")


def producer(queue: multiprocessing.Queue, n_consumers: int) -> None:
  for i in range(10):
    queue.put(i)
  for _ in range(n_consumers):
    queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumers = [
    multiprocessing.Process(target=consumer, args=(f"consumer {k}", queue))
    for k in range(2)
  ]
  producer_process = multiprocessing.Process(
    target=producer, args=(queue, len(consumers))
  )
  for process in [*consumers, producer_process]:
    process.start()
  for process in [*consumers, producer_process]:
    process.join()

## Performing Two Different Tasks in Parallel with a `Pool`

The following code submits two different computations to the same `Pool` with [`map_async`](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.pool.Pool.map_async), which returns immediately instead of waiting for the results like `map`: both computations therefore run at the same time on the pool's processes.

Rewrite it with [`concurrent.futures.ProcessPoolExecutor`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.ProcessPoolExecutor).

In [ ]:
import multiprocessing


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with multiprocessing.Pool() as pool:
    a_results = pool.map_async(a, range(10))
    b_results = pool.map_async(b, range(10))
    print(a_results.get())
    print(b_results.get())

In [ ]:
# Your code here

### Solution

In [ ]:
import concurrent.futures


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with concurrent.futures.ProcessPoolExecutor() as executor:
    a_futures = [executor.submit(a, i) for i in range(10)]
    b_futures = [executor.submit(b, i) for i in range(10)]
    print([future.result() for future in a_futures])
    print([future.result() for future in b_futures])